
# Projecto Zuber: An├ílise de Transporte em Chicago

## ­ƒøá´©Å Passo 1: Ingest├úo e Auditoria de Dados (Check-in)

O objetivo desta fase ├® carregar as bases de dados extra├¡das via SQL e garantir que a estrutura t├®cnica est├í pronta para an├ílise, mitigando riscos de corrup├º├úo de tipos ou valores ausentes.



In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Importa├º├úo
df_companies = pd.read_csv('C:/moved_project_sql_result_01.csv')
df_neighborhoods = pd.read_csv('C:/moved_project_sql_result_04.csv')
df_hypothesis = pd.read_csv('C:/moved_project_sql_result_07.csv')

In [ ]:
df_companies.info()

In [ ]:
df_companies.head()

In [ ]:
df_companies.isna().sum()

In [ ]:
df_companies.duplicated().sum()

In [ ]:
df_companies.describe()

In [ ]:
df_neighborhoods.info()

In [ ]:
df_neighborhoods.head()

In [ ]:
df_neighborhoods.duplicated().sum()

In [ ]:
df_neighborhoods.isna().sum()

In [ ]:
df_neighborhoods.describe()

In [ ]:
def audit_data(df, name):
    print(f"--- Auditoria: {name} ---")
    print(f"Dimens├Áes: {df.shape}")
    print("\nTipos de Dados e Nulos:")
    print(df.info())
    print("\nValores Duplicados:", df.duplicated().sum())
    print("\nEstat├¡stica Descritiva:")
    print(df.describe())
    print("-" * 30, "\n")
# Executando a auditoria inicial
audit_data(df_companies, "Empresas")
audit_data(df_neighborhoods, "Bairros (Destinos)")
audit_data(df_hypothesis, "Teste de Hip├│tese")

### ­ƒº╣ Passo 2: Prepara├º├úo e Limpeza dos Dados

In [ ]:
# --- LIMPEZA  ---

# 1. Convers├úo de data para o dataset de hip├│tese 
df_hypothesis['start_ts'] = pd.to_datetime(df_hypothesis['start_ts'])

# 2. Verifica├º├úo de valores negativos 
if (df_companies['trips_amount'] < 0).any():
    print("ALERTA: Detectados valores negativos em trips_amount. Removendo...")
    df_companies = df_companies[df_companies['trips_amount'] >= 0]

### ­ƒôè Fase 3: Identifica├º├úo dos 10 Principais Bairros

In [ ]:
#  10 principais destinos
top_10_neighborhoods = df_neighborhoods.sort_values(by='average_trips', ascending=False).head(10)

print("TOP 10 DESTINOS (NOVEMBRO 2017):")
display(top_10_neighborhoods)

### ­ƒôê Fase 4: Visualiza├º├úo e Conclus├Áes T├®cnicas

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 6))
sns.barplot(data=df_companies.sort_values(by='trips_amount', ascending=False).head(10), 
            x='trips_amount', y='company_name', hue='company_name', palette='Blues_d', legend=False)
plt.title('Top 10 Empresas por Volume de Corridas (15-16 Nov 2017)')
plt.show()

Conclus├úo : Existe uma discrep├óncia enorme entre o l├¡der (Flash Cab) e os demais. Isso indica que o mercado de Chicago ├® dominado por grandes frotas, dificultando a entrada de novos players (como a Zuber) a menos que haja um diferencial de pre├ºo ou tempo de espera.

Gr├ífico 2: Densidade Geogr├ífica (Destinos)

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(data=top_10_neighborhoods, x='average_trips', y='dropoff_location_name', hue='dropoff_location_name', palette='Reds_d', legend=False)
plt.title('Top 10 Bairros por M├®dia de Viagens Finalizadas (Nov 2017)')
plt.show()

Conclus├úo: O Loop e River North n├úo s├úo apenas destinos populares; eles s├úo outliers de demanda. A Zuber deve focar 80% de sua estrat├®gia de posicionamento de motoristas nestas zonas durante o hor├írio comercial.

### ­ƒº¬ Passo 5: Teste de Hip├│tese (Estat├¡stica de Produ├º├úo)

In [ ]:
import pandas as pd
from scipy import stats as st

# 1. Carga dos dados 
df_trips = pd.read_csv('C:/moved_project_sql_result_07.csv')

# 2. Segmenta├º├úo das Amostras para Teste
# Sele├º├úo de dura├º├Áes (em segundos) para as duas condi├º├Áes meteorol├│gicas
rainy_saturdays = df_trips[df_trips['weather_conditions'] == 'Bad']['duration_seconds']
good_saturdays = df_trips[df_trips['weather_conditions'] == 'Good']['duration_seconds']

# 3. Formula├º├úo das Hip├│teses
# H0 (Nula): A dura├º├úo m├®dia das viagens do Loop para o O'Hare ├® a mesma em s├íbados chuvosos e bons.
# H1 (Alternativa): A dura├º├úo m├®dia das viagens do Loop para o O'Hare muda nos s├íbados chuvosos.

# 4. Defini├º├úo do N├¡vel de Signific├óncia (Alpha)
# Defini├º├úo alpha em 0.05 (5%), .
alpha = 0.05 

# 5. Auditoria de Vari├óncia (Teste de Levene)
stat_levene, p_levene = st.levene(rainy_saturdays, good_saturdays)
equal_var_bool = p_levene > 0.05

# 6. Execu├º├úo do Teste t de Student para duas amostras independentes
results = st.ttest_ind(rainy_saturdays, good_saturdays, equal_var=equal_var_bool)

print(f"P-value: {results.pvalue}")

# 7.  Decis├úo Estat├¡stica
if results.pvalue < alpha:
    print("Conclus├úo: Rejeita a hip├│tese nula.")
else:
    print("Conclus├úo: N├úo podemos rejeitar a hip├│tese nula.")

# An├ílise Estrat├®gica ÔÇô Entrada da Zuber em Chicago

## 1. Contexto

Foi realizada uma an├ílise t├®cnica do mercado de transporte em Chicago com foco em:

- Estrutura competitiva  
- Concentra├º├úo geogr├ífica de demanda  
- Impacto das condi├º├Áes clim├íticas na efici├¬ncia operacional  

O objetivo foi identificar riscos operacionais e oportunidades estrat├®gicas para uma poss├¡vel entrada da Zuber no mercado.

---

## 2. Principais Achados

### Concentra├º├úo de Mercado
- A Flash Cab lidera com 19.558 viagens no per├¡odo analisado.
- O mercado apresenta forte concentra├º├úo e barreiras de escala relevantes.

### Concentra├º├úo Geogr├ífica
- Loop e River North s├úo os principais polos de destino.
- A demanda est├í fortemente centralizada nesses hubs financeiros.

### Impacto do Clima
- O teste estat├¡stico confirmou diferen├ºa significativa na dura├º├úo das viagens em dias chuvosos (p-value = 6,51e-12).
- Em condi├º├Áes adversas, o tempo m├®dio das corridas aumenta de forma relevante.

---

## 3. Impacto Financeiro e Operacional

### Efici├¬ncia da Frota
- O aumento da dura├º├úo m├®dia das viagens reduz o n├║mero de corridas por motorista por hora.
- Impacta diretamente a receita por hora e a margem operacional.

### Competitividade
- Alta concentra├º├úo de mercado exige diferencia├º├úo operacional (tempo de espera, disponibilidade e gest├úo din├ómica de oferta).

### Aloca├º├úo de Recursos
- Opera├º├úo dispersa reduz efici├¬ncia.
- Foco em hubs estrat├®gicos pode aumentar taxa de ocupa├º├úo e reduzir ETA.

---

## 4. Recomenda├º├úo

Recomenda-se avan├ºar com uma abordagem estruturada e controlada:

1. **Implementar teste piloto de precifica├º├úo din├ómica** em dias de clima adverso, para proteger margem e equilibrar oferta e demanda.
2. **Priorizar posicionamento estrat├®gico da frota** nos hubs de maior demanda (Loop e River North), especialmente em hor├írios comerciais.
3. **Monitorar indicadores-chave:** dura├º├úo m├®dia, corridas por hora, receita por motorista e impacto na margem.

---

### S├¡ntese Executiva

O mercado apresenta potencial relevante nos centros financeiros da cidade, por├®m exige efici├¬ncia operacional e mecanismos adaptativos para competir com players consolidados. A ado├º├úo de precifica├º├úo din├ómica e aloca├º├úo inteligente de frota aumenta a probabilidade de entrada sustent├ível e rent├ível.
